In [1]:
from google.colab import drive
import pandas as pd

# Mount Google Drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
# ===================== 1. IMPORTS & LOAD =====================
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                             ConfusionMatrixDisplay)

SEED = 42
DATA_DIR = "/content/drive/MyDrive/project"

X_raw = pd.read_csv(f"{DATA_DIR}/train_values.csv")
y_raw = pd.read_csv(f"{DATA_DIR}/train_labels.csv")
df = X_raw.merge(y_raw, on="building_id", validate="one_to_one")
print(df.shape, "| missing:", df.isna().sum().sum())

(260601, 40) | missing: 0


In [3]:
# ===================== 2. CLEANING: INVALID AGE =====================
AGE_MAX = 120
df["age_unknown"] = (df["age"] > AGE_MAX).astype(int)
print("Flagged ages:", df["age_unknown"].sum())
df.loc[df["age_unknown"] == 1, "age"] = np.nan



Flagged ages: 1734


In [4]:
# ===================== 3. SPLIT (stratified 70/15/15) =====================
X = df.drop(columns=["building_id", "damage_grade"])
y = df["damage_grade"]

X_train, X_tmp, y_train, y_tmp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=SEED)
X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=SEED)
print(y_train.value_counts(normalize=True).round(3).to_dict())
print(y_val.value_counts(normalize=True).round(3).to_dict())
print(y_test.value_counts(normalize=True).round(3).to_dict())

{2: 0.569, 3: 0.335, 1: 0.096}
{2: 0.569, 3: 0.335, 1: 0.096}
{2: 0.569, 3: 0.335, 1: 0.096}


In [5]:
# ===================== 4. PREPROCESSING (fit on train only) =====================
# 4a. frequency-encode the two high-cardinality geo columns using TRAIN frequencies
for col in ["geo_level_2_id", "geo_level_3_id"]:
    freq = X_train[col].value_counts(normalize=True)
    for d in (X_train, X_val, X_test):
        d[col + "_freq"] = d[col].map(freq).fillna(0)
X_train, X_val, X_test = [d.drop(columns=["geo_level_2_id", "geo_level_3_id"])
                          for d in (X_train, X_val, X_test)]


In [6]:
# 4b. cap outliers in skewed columns at the 1st/99th percentile (TRAIN quantiles)
skewed = ["age", "area_percentage", "height_percentage"]
lo, hi = X_train[skewed].quantile(0.01), X_train[skewed].quantile(0.99)
for d in (X_train, X_val, X_test):
    d[skewed] = d[skewed].clip(lower=lo, upper=hi, axis=1)

In [7]:
# 4c. column groups
plain_num = ["count_floors_pre_eq", "count_families", "geo_level_2_id_freq", "geo_level_3_id_freq"]
binary = [c for c in X_train.columns if c.startswith("has_")] + ["age_unknown"]
cat_cols = list(X_train.select_dtypes(exclude="number").columns) + ["geo_level_1_id"]
leftover = set(X_train.columns) - set(skewed + plain_num + binary + cat_cols)
print("Unassigned columns (should be empty):", leftover)

Unassigned columns (should be empty): set()


In [8]:
# 4d. impute -> log1p -> scale (skewed), scale (others), one-hot (categoricals)
ct_lr = ColumnTransformer([
    ("skewed", Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
        ("sc",  StandardScaler())]), skewed),
    ("num", StandardScaler(), plain_num),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
    ("bin", "passthrough", binary),
], verbose_feature_names_out=False)

Xtr = ct_lr.fit_transform(X_train)
Xva = ct_lr.transform(X_val)
Xte = ct_lr.transform(X_test)
names = ct_lr.get_feature_names_out()
print("Matrix shapes:", Xtr.shape, Xva.shape, Xte.shape, "| NaNs:", np.isnan(Xtr).sum())


Matrix shapes: (182420, 99) (39090, 99) (39091, 99) | NaNs: 0


In [9]:
# ===================== 5. BASELINE LOGISTIC REGRESSION =====================
def report(name, y_true, y_pred):
    print(f"\n--- {name} ---")
    print("Accuracy :", round(accuracy_score(y_true, y_pred), 4))
    print("Micro F1 :", round(f1_score(y_true, y_pred, average="micro"), 4))
    print("Macro F1 :", round(f1_score(y_true, y_pred, average="macro"), 4))
    print(classification_report(y_true, y_pred, digits=3))

t = time.time()
lr_base = LogisticRegression(max_iter=300, random_state=SEED)
lr_base.fit(Xtr, y_train)
print(f"Baseline train time: {time.time()-t:.0f}s")
pred_base = lr_base.predict(Xva)
report("LR baseline (validation)", y_val, pred_base)

Baseline train time: 43s

--- LR baseline (validation) ---
Accuracy : 0.6764
Micro F1 : 0.6764
Macro F1 : 0.5958
              precision    recall  f1-score   support

           1      0.621     0.357     0.454      3768
           2      0.680     0.822     0.744     22239
           3      0.679     0.521     0.590     13083

    accuracy                          0.676     39090
   macro avg      0.660     0.567     0.596     39090
weighted avg      0.674     0.676     0.664     39090



In [ ]:
# ===================== 6. HYPERPARAMETER TUNING =====================
idx = np.random.RandomState(SEED).choice(len(Xtr), min(50_000, len(Xtr)), replace=False)
X_s, y_s = Xtr[idx], y_train.to_numpy()[idx]

param_dist = {
    "C":            [0.001, 0.01, 0.1, 1, 10, 100],
    "l1_ratio":     [0, 0.5, 1],            # 0 = L2, 1 = L1, 0.5 = mix
    "class_weight": [None, "balanced"],
}
search = RandomizedSearchCV(
    LogisticRegression(penalty="elasticnet", solver="saga",
                       max_iter=200, tol=1e-3, random_state=SEED),
    param_dist, n_iter=12, cv=3, scoring="f1_macro",
    random_state=SEED, n_jobs=-1, verbose=2)
search.fit(X_s, y_s)

print("Best params:", search.best_params_)
print("Best CV macro F1:", round(search.best_score_, 4))
results = (pd.DataFrame(search.cv_results_).sort_values("rank_test_score")
           [["rank_test_score", "mean_test_score", "std_test_score", "params"]])
results.to_csv("lr_tuning_results.csv", index=False)
print(results.head(10).to_string())

Fitting 3 folds for each of 12 candidates, totalling 36 fits


In [ ]:
# ===================== 7. TUNED MODEL (refit on full train) =====================
lr_best = LogisticRegression(**search.best_params_, penalty="elasticnet",
                             solver="saga", max_iter=300, tol=1e-3, random_state=SEED)
lr_best.fit(Xtr, y_train)
pred_best = lr_best.predict(Xva)
report("LR tuned (validation)", y_val, pred_best)

comp = pd.DataFrame({
    "Model": ["LR baseline", "LR tuned"],
    "Accuracy": [accuracy_score(y_val, p) for p in (pred_base, pred_best)],
    "Micro F1": [f1_score(y_val, p, average="micro") for p in (pred_base, pred_best)],
    "Macro F1": [f1_score(y_val, p, average="macro") for p in (pred_base, pred_best)],
}).round(4)
print(comp.to_string(index=False))

In [ ]:
# ===================== 8. PLOTS =====================
ConfusionMatrixDisplay.from_predictions(y_val, pred_best, normalize="true", cmap="Blues")
plt.title("Logistic Regression (tuned) – normalised confusion matrix, validation set")
plt.savefig("lr_confusion_matrix.png", dpi=200, bbox_inches="tight")
plt.show()

coef = pd.Series(np.abs(lr_best.coef_).mean(axis=0), index=names).sort_values()
coef.tail(15).plot(kind="barh", figsize=(7, 5),
                   title="Top 15 features by mean |coefficient| – Logistic Regression")
plt.xlabel("Mean absolute coefficient")
plt.savefig("lr_coefficients.png", dpi=200, bbox_inches="tight")
plt.show()
print("Features zeroed out by L1:", int((coef == 0).sum()), "of", len(coef))

In [ ]:
# ===================== 9. FINAL TEST SCORE (run ONCE, after all tuning) =====================
pred_test = lr_best.predict(Xte)
report("LR tuned (TEST)", y_test, pred_test)